# 02 · Campaign — the systematic MPNN settings sweep

**Standard slot:** *design campaign.* **For Project 02 this means:** the campaign is a **parameter
sweep**, not novel backbone generation. You design across the grid
temperature {0.1, 0.2, 0.3, 0.5} × backbone noise {0.0, 0.1, 0.2} × seqs/backbone {8, 16, 48} for
each of your 20–30 backbones — **hundreds of sequences** — and write them to
`results/sequences.csv` (D2). The `mock` backend makes this run anywhere; the real ProteinMPNN
command is shown so you can reproduce it on Colab.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change)

ProteinMPNN/LigandMPNN are installed from source and **move** — pin a commit and verify the repo
still exists before a campaign. This cell HTTP-checks the pinned upstream URLs (it is allowed to
fail offline; on Colab it confirms the repos are reachable).

In [ ]:
import requests

# Pinned upstreams (verify + pin a COMMIT in your LOG.md before the course; these change):
#   ProteinMPNN  https://github.com/dauparas/ProteinMPNN   (pin e.g. a commit hash)
#   LigandMPNN   https://github.com/dauparas/LigandMPNN    (pin e.g. a commit hash)
UPSTREAMS = {
    "ProteinMPNN": "https://github.com/dauparas/ProteinMPNN",
    "LigandMPNN":  "https://github.com/dauparas/LigandMPNN",
}
for name, url in UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=10)
        print(f"{name:12s} {url}  -> HTTP {r.status_code}")
    except Exception as e:
        print(f"{name:12s} {url}  -> could not reach ({e}); fine offline, re-check on Colab")
print("\nReminder: pin a COMMIT (not just the repo) and log it — APIs/flags drift between commits.")

## 1 · The sweep grid

Sweep every (temperature, noise, seqs) cell across all backbones. With 20–30 backbones this is
hundreds of sequences. The `mock` backend is deterministic so the loop is reproducible; switch
`TOOL` to `"proteinmpnn"` on Colab.

In [ ]:
import itertools, pandas as pd
from mpnn_tools import run_mpnn

TEMPS  = [0.1, 0.2, 0.3, 0.5]
NOISES = [0.0, 0.1, 0.2]
NSEQS  = [8, 16, 48]
TOOL   = "mock"     # -> "proteinmpnn" on Colab (pin the commit)

# A small stand-in backbone set so the notebook runs end-to-end. REPLACE with your 20-30
# de novo backbones (Project 03 outputs / public design set) dropped into data/inputs/.
BACKBONES = [f"demo_backbone_{i:02d}" for i in range(1, 6)]   # 5 for the dry run; use 20-30 for real
print(f"backbones: {len(BACKBONES)}   grid cells: {len(TEMPS)*len(NOISES)*len(NSEQS)}")

## 2 · Run the sweep → `results/sequences.csv`

Note the **compute budget**: MPNN itself is seconds per backbone (T4 or even CPU). The cost comes
later in notebook 03/04 when you *recapitulate* each sequence with ESMFold/AF2 — so generate broadly
here, then triage. Every row carries its full setting.

In [ ]:
rows = []
for bb in BACKBONES:
    for temp, noise, nseq in itertools.product(TEMPS, NOISES, NSEQS):
        for d in run_mpnn(bb, temperature=temp, noise=noise, n_seqs=nseq, tool=TOOL):
            rows.append(d.as_row())

seqs = pd.DataFrame(rows)
# recapitulation columns (scrmsd, plddt) are filled in nb 03/04 after prediction.
seqs.to_csv("results/sequences.csv", index=False)
print("wrote results/sequences.csv", seqs.shape)
print(seqs[["backbone", "temperature", "noise", "n_seqs", "seq_index",
            "net_charge", "hydrophobic_fraction", "camsol_like"]].head())

### Note on scale (real run)

With 5 demo backbones you already have hundreds of rows because of the seqs/backbone axis. For the
real D2, use 20–30 backbones and the `proteinmpnn` backend. Budget recapitulation, not generation:
recapitulate with ESMFold first (seconds/seq), reserve AF2 full-MSA for survivors, batch overnight.
This is the genuine free-tier bottleneck — see `MANUAL.md §1/§6`.

## 3 · The real ProteinMPNN command (for reproducibility)

The mock backend stands in for this documented call. Pin the commit; log the exact flags.

In [ ]:
REAL_CMD = r"""
# Clone + pin a commit first:
#   git clone https://github.com/dauparas/ProteinMPNN && cd ProteinMPNN && git checkout <COMMIT>
# Then, per backbone × setting:
python ProteinMPNN/protein_mpnn_run.py \
    --pdb_path data/inputs/backbone_001.pdb \
    --out_folder results/mpnn/backbone_001 \
    --num_seq_per_target 16 \
    --sampling_temp "0.2" \
    --backbone_noise "0.1" \
    --seed 37
# Parse results/mpnn/backbone_001/seqs/*.fa into the same schema as results/sequences.csv.
"""
print(REAL_CMD)

## D2 checklist
- [ ] Version-verify cell run; ProteinMPNN/LigandMPNN commit pinned in `LOG.md`.
- [ ] `results/sequences.csv`: full grid × 20–30 backbones, every setting on every row (hundreds of seqs).
- [ ] Design log (settings, seeds, tool versions, runtimes) in `LOG.md`.
- [ ] Compute-budget note: MPNN seconds vs recapitulation cost; your triage plan.
- [ ] 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared filter on the swept sequences.